# Transaction Foundation Model on Ray — Part 1: Setup

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

### Anyscale Technical Demo — Ray Data + Ray Train + Ray Serve

---

## What we're building

Transaction foundation models are the latest generation of transformer models — like LLMs, but instead of language, they are trained on financial transactions and similar timeseries record data. This lets them recognize patterns, like fraud, that traditional ML techniques can't detect. Stripe, Visa, Mastercard, Nubank, and Revolut all run models like this in production.

This series of notebooks shows you how to build your own transaction foundation model, with performance and scalability that surpass [NVIDIA's blueprint](https://github.com/NVIDIA-AI-Blueprints/transaction-foundation-model), using an identical approach but with Ray achieving a scale that the NVIDIA blueprint cant reach. Here, we reuse NVIDIA's tokenizer, train their exact architecture with their recipe, and score fraud with their same evaluation method. So its an apples-to-apples comparison. Ray is the only difference. Along the way we maximize our GPUs, solve bottlenecks through independently scalable CPUs, and save money using a fault-tollerant cluster of AWS Spot instances. 

This is what a production workload looks like. In comparison, NVIDIA's repo runs every stage on a single node, and it never actually trains the foundation model — its tains a small 30 steps demonstration, and then downloads the real weights. This template runs the actual training: ~16,000 steps, or 8 epochs over all 19.5 million training transactions, about two hours on eight GPUs. Dont worry, with the exact same set of notebooks, you can scale down just as easily as you scale up - see our 'mini.yaml' config file for a small cpu-only configuration of this exact same code.

## 

## How we're building it

This series builds a complete fraud detection system, showcasing the alternative approaches a team can take with a transaction foundation model.

In this Part 1 notebook, we get oriented and set up the cluster and dependencies. Part 2 downloads the dataset and prepares it for training and testing. The dataset is IBM's TabFormer benchmark, a 24 million transaction credit card purchase database. Each transaction comes with 13 raw fields (amount, merchant, location, time, and so on). Those raw fields are what a typical fraud team uses in their traditional XGBoost based classifiers today.

Parts 3 & 4 pretrain the **foundation model** itself: a transformer trained on 19.5 million training transactions to predict each card's next transaction. This unlabeled training allows us to build a model that learns the data's underlying patterns without having to manually label transactions as fraud (or not). This is the same way an LLM pre-trains on large volumes of plain natural language text.

After the pre-training, the rest of the series builds **five fraud detectors**, each using the foundation model in more powerful ways than the last.

1. **raw** (Part 6) — XGBoost on the 13 raw fields alone. No foundation model involved. This is the 'traditional' machine learning approach, and the baseline to beat. This is identical to NVIDIA's blueprint to show an apples-to-apples comparison with later approaches.
2. **embedding** (Parts 5 & 6) — XGBoost on the foundation model's **embedding**: run a transaction through the model and it returns 512 numbers encoding what the model understands about that transaction. This is the most straightforward way for XGBoost to use a foundation model. Part 5 performs the embeddings, Part 6 classifies fraud with them. 
3. **fusion** (Part 6) — This is an ensemble approach, XGBoost uses the traditional "raw" fields, and also the embedding. The idea here is that these two models identify fraud in different ways, combined they are more powerful.
4. **fine-tuned** (Part 7) — Finally, removing XGBoost entirely, basing the model purely on a Transformer model. We put a classification head on the foundation model and train its weights on the fraud labels directly, the way an LLM gets fine-tuned for a task. The most modern approach here. If this approach beats our fusion model, it allows an organization to completely remove the fragile, ad-hoc, manually built xgboost based pipeline.
5. **fine-tuned + raw** (Part 7) — an ensemble of the fine-tuned model's score with the raw fields, for teams that keep their existing pipeline. It scores highest of all, but it retains exactly the xgboost pipeline the fine-tuned approach exists to remove.

One foundation model, five detectors — using it not at all, as a feature factory, as extra features, as the classifier itself, and as an ensemble with what you already have. Every performance number in this series is one of these five scored on the same held-out transactions.

## Model performance

Every model outputs a fraud score for each transaction. We evaluate how well the model performs by evaluating two metrics: **Average Precision (AP)** and **AUC-ROC**. AP measures how often the model is correct when it alerts that a transaction is fraudulent, averaged over every possible alert volume — a perfect model scores 1.0, a random one scores 0.001 here. AUC-ROC measures how reliably fraud scores higher than normal transactions. Its differences are real but compressed — every model here lands between 0.92 and 0.99, and a move from 0.989 to 0.992 quietly removes about a third of the remaining ranking errors. It also counts an error the same wherever it happens in the ranking, while AP weights the top, where alerts actually get raised. That's why AP spreads the same two models much further apart (raw vs the fine-tuned ensemble: AUC-ROC 0.989 vs 0.992, AP 0.124 vs 0.199). We compare on AP; it's also the metric NVIDIA reports.

**Average precision on NVIDIA's 100K-transaction test set:**

| model | NVIDIA AP | our AP | notes |
|---|---|---|---|
| raw | 0.1238 | 0.1238 | the control: their exact features and recipe, so it must match — and does |
| embedding | 0.0123 | 0.04–0.06 | our foundation model's embedding, 3–5× theirs |
| fusion | 0.1755 | 0.136 † | noisy at ~112 test frauds; our best draw is 0.284 |
| fine-tuned | — | 0.1263 | no raw pipeline at all; parity with raw |
| fine-tuned + raw | — | 0.1988 | best score in the series, but keeps the raw pipeline |

Three results matter here. Our foundation model's embedding scores 3–5× higher than theirs. The fine-tuned model — pure transformer, no XGBoost, no raw feature pipeline — matches the raw baseline on its own (a 5,000-resample bootstrap puts them dead even), which means an organization could retire the hand-built pipeline entirely without losing accuracy. And the raw row matching exactly is the control that proves the wins are the model, not a difference in our method.

† Fusion is the noisy one: with only ~112 frauds in the test set, any single fusion score is largely luck of the draw. NVIDIA's 0.1755 is one draw; 0.136 is our median across many draws, and we clear their number in about 1 in 6 draws. Part 6 reports the full distribution.

## Scalability

This template distributes each stage on the hardware it needs: data preparation on autoscaling CPU workers with Ray Data, pretraining on eight GPUs with Ray Train, and embedding extraction on both at once — CPU workers feed transactions to GPU workers running the model. The CPU and GPU pools scale independently, so the GPUs aren't stuck waiting on data work. NVIDIA's pipeline runs all of this on one machine.

Distributing a pipeline can change its results without you noticing. We checked: every distributed stage produces output identical to NVIDIA's single-machine version — same 19.5 million rows in the same order, bit-for-bit identical training corpus, byte-equal labels and features. The checks live in `scripts/verify_*.py`, and the matched raw baseline above is the end-to-end proof.

Cost-wise, workers come up when a stage asks for them (2–3 minutes in our runs) and scale back to zero after, so the GPUs only exist for the ~2 hours that use them. A single always-on A100 node pays for its GPU the whole time.

## Architecture

```
                     ┌─────────────────────── Anyscale ────────────────────────┐
 IBM TabFormer ────► │ [Ray Data]   temporal split + corpus (CPU workers)      │
 (24M txns)          │ [Ray Train]  next-token pretraining, 8 GPUs (DDP)       │
                     │ [Ray Data]   embeddings: CPU tokenize → GPU actors      │
                     │ [XGBoost]    fraud: raw vs embedding vs fusion (GPU)    │
                     │ [Ray Train]  fine-tune: fraud head on the model, 8 GPUs │
                     │ [Ray Serve]  online embedding + fraud score             │
                     └─────────────────────────────────────────────────────────┘
```

Every stage is the **same code** from a single machine to a multi-node cluster — you change `ScalingConfig`, not your program. One `SCALE` variable at the top of each notebook selects a config file under `configs/`, and diffing two of those files shows the complete difference between two scales.

## The series

We build the foundation model end-to-end, one stage per notebook:

| # | Notebook | Runs on |
|---|----------|---------|
| **1** | **Setup** *(this notebook)* | — |
| 2 | Load & explore the data | Ray Data, CPU workers |
| 3 | Tokenize — build the pretrain corpus | Ray Data, CPU workers |
| 4 | Pretrain — next-token prediction | Ray Train, 8 GPUs |
| 5 | Batch embedding extraction | Ray Data, CPU → GPU actors |
| 6 | Downstream fraud — raw vs embedding vs fusion | XGBoost on GPU |
| 7 | Fine-tune the foundation model | Ray Train, 8 GPUs |
| 8 | Online serving | Ray Serve |
| 9 | Run the whole pipeline as a job | Anyscale Jobs |
| 10 | Scaling up — the bottlenecks you'll hit | All of the above |

## What a full run needs

`mini` proves the plumbing on CPU in minutes. The results above come from `SCALE = "full"`, which needs:

- the IBM TabFormer dataset — a one-time ~2.3 GB download that Part 2 handles,
- GPU workers — 8×A10G for the ~2h pretraining, for embedding extraction, and for about an hour of Part 7's fine-tuning (two variants, ~30 min each); the downstream XGBoost also needs a GPU (Part 6 explains why the result depends on it),
- a CPU worker group in the cluster config, so the CPU-heavy data stages don't pull up GPU nodes just for their CPUs (Part 10 shows that failure mode).

Nothing else changes between the two scales. Same notebooks, same code, different config file.

---

## Step 1: Install dependencies

The cell below installs the template's dependencies and registers them on every cluster node, so the same imports resolve on workers as well as the head node. Two pins matter. `xgboost` is 3.2.0 because the downstream fraud result is sensitive to an early-stopping behavior that changed in 3.3. RAPIDS (`cudf`) is NVIDIA's GPU tokenizer, kept as the reference implementation — the notebooks run a CPU implementation we verified byte-identical to it, which is why `mini` needs no GPU at all.

> In production you'd install from the generated `python_depset.lock`. Here we install from `requirements.txt` for portability.

In [1]:
!pip install -q -r requirements.txt

Successfully registered `ray, torch` and 13 other packages to be installed on all cluster nodes.
View and update dependencies here: https://console.anyscale.com/cld_g54aiirwj1s8t9ktgzikqur41k/prj_f1j47h9srml4cyg962id75ms2e/workspaces/expwrk_78mtwtucrd61tjxf851krarzwr?workspace-tab=dependencies


## Step 2: Attach to the cluster

In an Anyscale Workspace, Ray is already running — `ray.init()` **attaches** to the workspace's cluster rather than starting one. `working_dir` ships this template's `src/` package to every worker, so the notebook and remote tasks/actors all import the same code.

In [2]:
import sys, os
import logging


# Make the template's `src/` package importable from the notebook.
DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import ray

# In an Anyscale Workspace, Ray is already running — this attaches to it.
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

from src.utils import print_cluster_resources
print_cluster_resources()

Ray cluster resources:
  anyscale/cpu_only:true 1.0
  anyscale/node-group:head 1.0
  anyscale/provider:aws 1.0
  anyscale/region:us-west-2 1.0
  memory               34359738368.0
  object_store_memory  9599240601.0

Cluster nodes: 1
  10.0.158.37          alive=True  anyscale/cpu_only:true=1.0, anyscale/region:us-west-2=1.0, anyscale/provider:aws=1.0, memory=34359738368.0, object_store_memory=9599240601.0, anyscale/node-group:head=1.0


/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


You should see the head node and its resources. On a fresh or idle cluster the head node intentionally schedules no work — GPU and CPU worker nodes autoscale up when later stages request them, and scale back down when idle. You don't manage that; Ray does.

That's the whole setup: dependencies registered, cluster attached. Every notebook from here starts the same way and picks up where the previous one left off, using artifacts written to shared storage.

---

## Next

**Part 2 — Load & explore the data**: download the IBM TabFormer benchmark, rebuild NVIDIA's temporal train/val/test split from the raw CSV with Ray Data, and see in the data why a 0.1% fraud rate dictates how the whole series measures success.